# 🐾 AnimalMind — ViT Dog Breed Classifier Training Pipeline (>90% Target Accuracy)

This notebook trains and calibrates the ViT dog breed classifier on **Stanford Dogs (120 breeds)** with:
- **Data Augmentations**: RandAugment(num_ops=2, magnitude=9), ColorJitter, MixUp, CutMix.
- **Regularization**: Label Smoothing (0.1), EMA, Stochastic Depth (0.2).
- **Uncertainty Calibration**: Temperature Scaling via L-BFGS to minimize Expected Calibration Error (ECE).
- **Model Export**: Automatic upload to Hugging Face Hub (`firstoff/animalmind-breed-classifier`).

In [ ]:
# 1. Verify GPU Acceleration
!nvidia-smi

In [ ]:
# 2. Clone Repository & Install Training Dependencies
!git clone https://github.com/firstoff23/AnimalMind.git
%cd AnimalMind/ml_backend
!pip install -q -r requirements_training.txt

In [ ]:
# 3. Configure Hugging Face Token (Optional - for direct HF Hub upload)
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("🔑 Hugging Face Token loaded successfully.")
except Exception:
    print("ℹ️ No Colab Secret 'HF_TOKEN' found. Model will be saved locally.")

In [ ]:
# 4. Launch 50-Epoch GPU Training Pipeline
!python -m training.train_dog_breeds \
    --epochs 50 \
    --batch-size 32 \
    --lr 3e-4 \
    --model-name google/vit-base-patch16-224 \
    --output-dir models/animalmind-breed-classifier \
    --push-to-hub firstoff/animalmind-breed-classifier

In [ ]:
# 5. Display Calibration & Metrics Report
import json
import torch

metrics_path = "training/training_metrics.json"
with open(metrics_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"🏆 Best Validation Accuracy : {data.get('best_val_accuracy', 0)*100:.2f}%")
print(f"🌡️ Calibrated Temperature T : {data.get('calibrated_temperature', 1.0):.4f}")

temp_data = torch.load("models/temperature.pt")
print("Saved Temperature Tensor:", temp_data)